In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
# Copyright 2017 The TensorFlow Authors All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# =============================================================================


import tensorflow as tf

from migration.config_old import TrainingSettings
from migration.models import vrnn
import migration.datasets as datasets

from migration.models.vrnn_elbo import VRNN

from pathlib import Path
# get batch and model
def create_dataset_and_model(config, shuffle, repeat):

    inputs, targets, _, _, _, lengths, mean =  datasets.create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                    '../../data/ct_2017010203_10_20/mean.pkl',
                    32, # batch size
                    99999, # not used lol
                    300,
                    300, 
                    30,
                    72, 
                    shuffle=False,
                    repeat=False)
    # Convert the mean of the training set to logit space so it can be used to
    # initialize the bias of the generative distribution.
    generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
    generative_distribution_class = vrnn.ConditionalBernoulliDistribution
    model = VRNN(inputs.get_shape().as_list()[2],
                             config.latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5, num_samples=1)
    return inputs, targets, lengths, model




def run_train(config):

    if config.random_seed: tf.random.set_seed(config.random_seed)

    inputs, targets, lengths, model = create_dataset_and_model(config,
                                                               shuffle=True,
                                                               repeat=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=config.learning_rate)
    
    @tf.function
    def train_step(x,y):
        with tf.GradientTape() as tape:
            bound = model((x, y),lengths)
            # Compute lower bounds on the log likelihood.
            bound = tf.reduce_mean(input_tensor=bound / tf.cast(lengths, dtype=tf.float32))
            loss = -bound
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        return loss
    ckpt = './here.weights.h5'

    for epoch in range(50):
        if Path(ckpt).exists():
            model.load_weights(ckpt)
        print(inputs.dtype)
        print(targets.dtype)
        loss_value = train_step(inputs, targets)
        print(loss_value)
        model.save_weights(ckpt, overwrite=True)


In [9]:

config = TrainingSettings()
# fh = logging.FileHandler(os.path.join(config.logdir,config.log_filename+".log"))
# # get TF logger
# logger = logging.getLogger('tensorflow')
# logger.addHandler(fh)
run_train(config)


<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.866514, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.87083, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.847748, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.78193, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.773994, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.77843, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.766155, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.745588, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.764652, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.752308, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.750639, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.739548, shape=(), dtype=floa

In [ ]:

config = TrainingSettings()
# fh = logging.FileHandler(os.path.join(config.logdir,config.log_filename+".log"))
# # get TF logger
# logger = logging.getLogger('tensorflow')
# logger.addHandler(fh)
c
run_train(config, )


<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.947773, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.878922, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.868382, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.824997, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.78374, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.76966, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.754204, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.754053, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.733292, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.732044, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.72961, shape=(), dtype=float32)
<dtype: 'float32'>
<dtype: 'float32'>
tf.Tensor(20.71307, shape=(), dtype=float

KeyboardInterrupt: 